# Team Season - common_all_players

This notebook inspects the data **downloaded** from the `common_all_players` endpoint.

Source folder: `02_data/01_raw/2025_26/02_team_season/common_all_players`

Scope:
- Read parquet files in the folder
- Show how many files were downloaded and how many rows/columns they contain
- Report missing values (null cells) overall and by row/column distribution

Notes:
- Read-only analysis: no exports or writes


In [17]:
from pathlib import Path
import pandas as pd
import numpy as np

# Resolve project root by walking up to find 02_data
project_root = Path.cwd()
for _ in range(6):
    if (project_root / "02_data").exists():
        break
    project_root = project_root.parent

endpoint = "common_all_players"
data_dir = project_root / "02_data" / "01_raw" / "2025_26" / "02_team_season" / endpoint

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)


In [18]:
files = sorted(data_dir.glob("*.parquet"))
print(f"Endpoint: {endpoint}")
print(f"Folder: {data_dir}")
print(f"Parquet files: {len(files)}")

files_df = pd.DataFrame({
    "file": [f.name for f in files],
    "size_mb": [round(f.stat().st_size / 1e6, 3) for f in files],
})
files_df


Endpoint: common_all_players
Folder: /Users/pablodiazgonzalez/Documents/MachineLearning/AlgoBasket/02_data/01_raw/2025_26/02_team_season/common_all_players
Parquet files: 1


,file,size_mb
0,common_all_players__season=2025-26.parquet,0.049


In [19]:
dfs = [pd.read_parquet(f) for f in files]

per_file = pd.DataFrame({
    "file": [f.name for f in files],
    "rows": [len(df) for df in dfs],
    "cols": [df.shape[1] for df in dfs],
})
per_file


,file,rows,cols
0,common_all_players__season=2025-26.parquet,561,18


In [20]:
df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

print(f"Combined shape: {df.shape}")
print(f"Total rows: {len(df)}")
print(f"Total columns: {df.shape[1] if len(df.columns) else 0}")


Combined shape: (561, 18)
Total rows: 561
Total columns: 18


In [21]:
rows, cols = df.shape
total_cells = rows * cols
null_cells = int(df.isna().sum().sum()) if total_cells else 0
null_pct = (null_cells / total_cells) if total_cells else 0

summary = pd.DataFrame({
    "rows": [rows],
    "cols": [cols],
    "total_cells": [total_cells],
    "null_cells": [null_cells],
    "null_pct": [round(null_pct * 100, 3)],
})
summary


,rows,cols,total_cells,null_cells,null_pct
0,561,18,10098,36,0.357


In [22]:
if df.empty:
    col_summary = pd.DataFrame()
else:
    col_nulls = df.isna().sum()
    col_null_pct = df.isna().mean()
    col_summary = (
        pd.DataFrame({
            "null_cells": col_nulls,
            "null_pct": (col_null_pct * 100).round(3),
        })
        .sort_values("null_pct", ascending=False)
    )

col_summary


,null_cells,null_pct
TEAM_SLUG,36,6.417
PERSON_ID,0,0.000
DISPLAY_LAST_COMMA_FIRST,0,0.000
SEASON,0,0.000
OTHERLEAGUE_EXPERIENCE_CH,0,0.000
GAMES_PLAYED_FLAG,0,0.000
TEAM_CODE,0,0.000
TEAM_ABBREVIATION,0,0.000
TEAM_NAME,0,0.000
TEAM_CITY,0,0.000


In [23]:
if df.empty:
    row_dist = pd.DataFrame()
else:
    row_null_pct = df.isna().mean(axis=1)
    bins = [0, 0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 1.0]
    row_bins = pd.cut(row_null_pct, bins=bins, include_lowest=True)
    row_counts = row_bins.value_counts().sort_index()
    row_dist = pd.DataFrame({
        "rows": row_counts,
        "pct_rows": (row_counts / len(row_null_pct) * 100).round(3),
    })

row_dist


,rows,pct_rows
"(-0.001, 0.01]",525,93.583
"(0.01, 0.05]",0,0.000
"(0.05, 0.1]",36,6.417
"(0.1, 0.25]",0,0.000
"(0.25, 0.5]",0,0.000
"(0.5, 0.75]",0,0.000
"(0.75, 1.0]",0,0.000


In [24]:
if df.empty:
    col_dist = pd.DataFrame()
else:
    col_null_pct = df.isna().mean()
    bins = [0, 0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 1.0]
    col_bins = pd.cut(col_null_pct, bins=bins, include_lowest=True)
    col_counts = col_bins.value_counts().sort_index()
    col_dist = pd.DataFrame({
        "columns": col_counts,
        "pct_columns": (col_counts / len(col_null_pct) * 100).round(3),
    })

col_dist


,columns,pct_columns
"(-0.001, 0.01]",17,94.444
"(0.01, 0.05]",0,0.000
"(0.05, 0.1]",1,5.556
"(0.1, 0.25]",0,0.000
"(0.25, 0.5]",0,0.000
"(0.5, 0.75]",0,0.000
"(0.75, 1.0]",0,0.000


In [25]:
# Remove invalid team rows and export
# Criteria: TEAM_ID == 0 or TEAM_SLUG is null/empty

if df.empty:
    print("No data to filter.")
else:
    team_id_col = "TEAM_ID" if "TEAM_ID" in df.columns else None
    team_slug_col = "TEAM_SLUG" if "TEAM_SLUG" in df.columns else None

    if team_id_col is None or team_slug_col is None:
        print("Required columns not found in dataframe.")
        print(df.columns)
    else:
        before_rows = len(df)
        mask = (df[team_id_col] == 0) | (df[team_slug_col].isna()) | (df[team_slug_col].astype(str).str.strip() == "")
        df_filtered = df.loc[~mask].copy()
        after_rows = len(df_filtered)

        print(f"Rows before: {before_rows}")
        print(f"Rows after: {after_rows}")
        print(f"Removed: {before_rows - after_rows}")

        out_dir = project_root / "02_data" / "02_cleaned" / "2025_26" / "02_team_season"
        out_dir.mkdir(parents=True, exist_ok=True)
        out_path = out_dir / "common_all_players.parquet"
        df_filtered.to_parquet(out_path, index=False)
        print(f"Saved to: {out_path}")


Rows before: 561
Rows after: 525
Removed: 36
Saved to: /Users/pablodiazgonzalez/Documents/MachineLearning/AlgoBasket/02_data/02_cleaned/2025_26/02_team_season/common_all_players.parquet
